# Building Trusted GeoAI

This notebook continues from `SpatialGrounding.ipynb`: it takes the ranked
recommendations generated there, runs them through a human-in-the-loop
review step with a durable audit trail (`traffic_safety.governance`), and
demonstrates swapping the recommendation pipeline to a local, sovereign
model (`OllamaLLMClient`) with no other code changes.

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import os

from arcgis.gis import GIS

from traffic_safety.recommendations import get_risk_layer, rank_recommendations
from traffic_safety.governance import (
    OllamaLLMClient,
    ReviewDecision,
    load_audit_trail,
    request_human_review,
)

In [ ]:
home_portal = GIS("home")
risk_layer_item_id = os.getenv("RISK_LAYER_ITEM_ID")
risk_layer = get_risk_layer(home_portal, risk_layer_item_id)
audit_log_path = os.getenv("AUDIT_LOG_PATH", "audit_trail.jsonl")

## Local, sovereign inference

`OllamaLLMClient` implements the exact same `generate(prompt)` interface used
by `TemplateLLMClient` in the previous session. Requires a local Ollama
server (`ollama serve`) with the configured model pulled, and the optional
`local-inference` dependency group (`uv sync --extra local-inference`).

In [ ]:
local_llm_client = OllamaLLMClient(model=os.getenv("OLLAMA_MODEL", "llama3.2"))
recommendations = rank_recommendations(risk_layer, top_n=5, llm_client=local_llm_client)

## Human-in-the-loop review

Every recommendation is reviewed before any downstream action is taken. The
reviewer sees the full evidence package (risk score, confidence range, years
observed, supporting context) and records approve / reject / needs-review.
`request_human_review` persists the decision as an append-only audit
record.

In [ ]:
reviewer = os.getenv("REVIEWER_NAME", "demo-reviewer")

for recommendation in recommendations:
    print(recommendation["recommendation"])
    answer = input("Approve this recommendation? [y/n/skip]: ").strip().lower()
    decision = {
        "y": ReviewDecision.APPROVED,
        "n": ReviewDecision.REJECTED,
    }.get(answer, ReviewDecision.NEEDS_REVIEW)

    request_human_review(recommendation, decision=decision, reviewer=reviewer, audit_log_path=audit_log_path)

## Audit trail

Every review decision is recorded alongside the full recommendation and
evidence that produced it, giving a transparent, reproducible record of who
approved what, and why.

In [ ]:
load_audit_trail(audit_log_path)